# Phase 2 (suite) — QSAR par empreintes moléculaires (Morgan fingerprints)

Le modèle de `02_rdkit_descriptors_qsar.ipynb` n'utilise que des descripteurs physico-chimiques globaux (poids, LogP, TPSA...). C'est lisible mais grossier : deux molécules avec le même poids moléculaire peuvent avoir des structures complètement différentes. Les **empreintes de Morgan** (ECFP) encodent la présence de sous-structures locales précises — c'est le standard du domaine en QSAR/virtual screening.

Ce notebook compare, en validation croisée, trois représentations : descripteurs seuls, empreintes seules, et la combinaison des deux — pour choisir le modèle final avec un chiffre à l'appui plutôt qu'un choix arbitraire.

In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdFingerprintGenerator
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier

df = pd.read_csv("../data/raw/erbb2_activities.csv")
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

DESCRIPTOR_COLS = [
    "molecular_weight", "logp", "h_bond_donors", "h_bond_acceptors",
    "rotatable_bonds", "tpsa", "aromatic_rings", "ring_count",
    "heavy_atoms", "molar_refractivity",
]

def compute_descriptors(mol):
    return [
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Lipinski.NumHDonors(mol),
        Lipinski.NumHAcceptors(mol), Descriptors.NumRotatableBonds(mol), Descriptors.TPSA(mol),
        Descriptors.NumAromaticRings(mol), Descriptors.RingCount(mol), Descriptors.HeavyAtomCount(mol),
        Descriptors.MolMR(mol),
    ]

desc_rows, fp_rows, valid_idx = [], [], []
for i, smi in enumerate(df["canonical_smiles"]):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    desc_rows.append(compute_descriptors(mol))
    fp_rows.append(np.array(morgan_gen.GetFingerprint(mol)))
    valid_idx.append(i)

X_desc = np.array(desc_rows)
X_fp = np.array(fp_rows)
X_combined = np.hstack([X_desc, X_fp])
y = df.loc[valid_idx, "active"].values
smiles_valid = df.loc[valid_idx, "canonical_smiles"].values

print("Descripteurs :", X_desc.shape, "| Empreintes :", X_fp.shape, "| Combiné :", X_combined.shape)

Descripteurs : (1070, 10) | Empreintes : (1070, 1024) | Combiné : (1070, 1034)


## 1. Comparaison en validation croisée (5-fold, ROC AUC)

In [2]:
model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    eval_metric="logloss", random_state=42,
)
cv = StratifiedKFold(5, shuffle=True, random_state=42)

results = {}
for name, X in [("descripteurs", X_desc), ("empreintes", X_fp), ("combiné", X_combined)]:
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    results[name] = scores
    print(f"{name:15s} ROC AUC = {scores.mean():.4f} +/- {scores.std():.4f}")

descripteurs    ROC AUC = 0.9545 +/- 0.0151


empreintes      ROC AUC = 0.9747 +/- 0.0083


combiné         ROC AUC = 0.9803 +/- 0.0073


## 2. Entraînement du modèle final (représentation combinée)

Descripteurs + empreintes obtient le meilleur score et reste interprétable en partie (les 10 premières colonnes restent des descripteurs physico-chimiques nommés, les 1024 suivantes sont les bits d'empreinte).

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

final_model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    eval_metric="logloss", random_state=42,
)
final_model.fit(X_train, y_train)

proba = final_model.predict_proba(X_test)[:, 1]
print("ROC AUC (hold-out) :", roc_auc_score(y_test, proba))
print(classification_report(y_test, final_model.predict(X_test)))

ROC AUC (hold-out) : 0.981214504150284
              precision    recall  f1-score   support

           0       0.92      0.98      0.95       105
           1       0.98      0.92      0.95       109

    accuracy                           0.95       214
   macro avg       0.95      0.95      0.95       214
weighted avg       0.95      0.95      0.95       214



## 3. Importance des variables

Les descripteurs physico-chimiques nommés restent-ils utiles à côté des 1024 bits d'empreinte ? On regarde leur rang dans l'importance globale du modèle.

In [4]:
feature_names = DESCRIPTOR_COLS + [f"fp_bit_{i}" for i in range(X_fp.shape[1])]
importances = pd.Series(final_model.feature_importances_, index=feature_names).sort_values(ascending=False)
importances.head(15)

fp_bit_343    0.353739
fp_bit_202    0.094235
fp_bit_593    0.048013
fp_bit_491    0.034867
ring_count    0.021835
fp_bit_871    0.021238
fp_bit_644    0.016319
fp_bit_964    0.012215
fp_bit_209    0.010623
fp_bit_946    0.010103
fp_bit_342    0.009392
fp_bit_222    0.008751
fp_bit_951    0.008471
fp_bit_694    0.008107
fp_bit_621    0.008000
dtype: float32

## 4. Fonction de screening avec le modèle final

In [5]:
def featurize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    desc = compute_descriptors(mol)
    fp = np.array(morgan_gen.GetFingerprint(mol))
    return np.concatenate([desc, fp]).reshape(1, -1)

def screen_molecule(smiles, model=final_model):
    features = featurize(smiles)
    if features is None:
        return {"error": "SMILES invalide"}
    proba = model.predict_proba(features)[0, 1]
    return {"activity_probability": round(float(proba), 3)}

# Mêmes médicaments de référence que le notebook 03, pour comparer les deux modèles
known_drugs = {
    "lapatinib": "CS(=O)(=O)CCNCC1=CC=C(O1)C2=CC3=C(C=C2)N=CN=C3NC4=CC(=C(C=C4)OCC5=CC(=CC=C5)F)Cl",
    "neratinib": "CCOC1=C(C=C2C(=C1)N=CC(=C2NC3=CC(=C(C=C3)OCC4=CC=CC=N4)Cl)C#N)NC(=O)/C=C/CN(C)C",
    "tucatinib": "CC1=C(C=CC(=C1)NC2=NC=NC3=C2C=C(C=C3)NC4=NC(CO4)(C)C)OC5=CC6=NC=NN6C=C5",
    "afatinib": "CN(C)C/C=C/C(=O)NC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC(=C(C=C3)F)Cl)O[C@H]4CCOC4",
    "pyrotinib": "CCOC1=C(C=C2C(=C1)N=CC(=C2NC3=CC(=C(C=C3)OCC4=CC=CC=N4)Cl)C#N)NC(=O)/C=C/[C@H]5CCCN5C",
}
for name, smi in known_drugs.items():
    print(name, screen_molecule(smi))

lapatinib {'activity_probability': 0.98}
neratinib {'activity_probability': 0.881}
tucatinib {'activity_probability': 0.991}
afatinib {'activity_probability': 0.988}
pyrotinib {'activity_probability': 0.893}


In [6]:
import joblib
joblib.dump(final_model, "../models/qsar_erbb2_combined_xgboost.pkl")
print("Modèle final sauvegardé : qsar_erbb2_combined_xgboost.pkl")

Modèle final sauvegardé : qsar_erbb2_combined_xgboost.pkl
